---
# Scénario 5 — Matrice des précipitations et de l’exposition aux inondations [CAS PRINCIPAL]
*Semaine 4* · Données : `S05_pluie_7jours.csv` (synthétiques)

### Contexte
Une semaine de fortes pluies a touché six communes côtières, de Gonaïves à Jacmel. Des stations météorologiques ont enregistré la pluie tombée chaque jour, et une enquête indique quelle part des ménages vit en zone inondable. L’équipe de réduction des risques ne peut pas envoyer d’inspecteurs partout en même temps : elle doit choisir où vérifier en premier les digues, les ravines et les abris. Vous travaillerez avec NumPy sur un tableau de chiffres, où chaque ligne est une commune et chaque colonne un jour.

## PAUSE — Discutez de la tâche analytique avant de toucher au clavier
Travaillez en binôme ou en petit groupe pendant **10 à 20 minutes**. Consignez les six réponses avant de coder.

**1. Quelle décision pourrait évoluer ?**  
_Votre réponse :_ _______________________________________________

**2. Qui est la partie prenante ?**  
_Votre réponse :_ _______________________________________________

**3. Que représente une ligne ?**  
_Votre réponse :_ _______________________________________________

**4. Quels champs et quelles unités sont nécessaires ?**  
_Votre réponse :_ _______________________________________________

**5. À quoi doit ressembler le résultat ?**  
_Votre réponse :_ _______________________________________________

**6. De quelles façons la réponse pourrait-elle être erronée ?**  
_Votre réponse :_ _______________________________________________

### Avant de coder, mettez-vous d’accord sur
- l’unité d’analyse et le grain d’une ligne ;
- les champs requis et leurs unités ;
- au moins deux contrôles de validation ;
- la forme du résultat final attendu.

> ⚠️ **Ce jeu de données contient au moins un piège propre à ce scénario.** Trouvez-le et expliquez-le avant de conclure.

In [11]:
# 1- Install (PIP) 2- LIBRAIRIES
import io
import pandas as pd
import numpy as np
from google.colab import files
from datetime import datetime, timedelta

In [12]:
# Téléverser le fichier
uploaded = files.upload()

Saving S05_pluie_7jours.csv to S05_pluie_7jours.csv


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
print(type(uploaded))
print(uploaded.keys())

<class 'dict'>
dict_keys(['S05_pluie_7jours.csv'])


In [16]:
pluie_df = pd.read_csv("S05_pluie_7jours.csv")
display(pluie_df)

,commune,j1,j2,j3,j4,j5,j6,j7,part_menages_zone_inondable
0,Gonaïves,12,18,44,96,72,31,20,0.35
1,Saint-Marc,10,14,38,88,69,27,17,0.22
2,Cabaret,8,16,51,110,83,42,25,0.41
3,Léogâne,6,11,29,75,58,24,15,0.30
4,Petit-Goâve,15,21,63,124,-999,48,32,0.38
5,Jacmel,9,12,35,81,64,30,19,0.18


In [17]:
#Tableau numpy des noms de communes
communes=pluie_df["commune"].to_numpy()

In [18]:
print(communes)
print(communes.shape)

['Gonaïves' 'Saint-Marc' 'Cabaret' 'Léogâne' 'Petit-Goâve' 'Jacmel']
(6,)


In [19]:
# Tableau NumPy 6 × 7 des pluies (colonnes j1 à j7), en nombres décimaux   # pluie = pluie_df[[f"j{i}" for  i  in  range  (1, 8)]].to_numpy().astype=float
pluie = pluie_df[[f"j{i}" for  i  in  range  (1, 8)]].to_numpy(dtype=float)
print(pluie)
print(pluie.shape)

[[  12.   18.   44.   96.   72.   31.   20.]
 [  10.   14.   38.   88.   69.   27.   17.]
 [   8.   16.   51.  110.   83.   42.   25.]
 [   6.   11.   29.   75.   58.   24.   15.]
 [  15.   21.   63.  124. -999.   48.   32.]
 [   9.   12.   35.   81.   64.   30.   19.]]
(6, 7)


In [20]:
# Trois contrôles :  forme, type, min
print("forme:",pluie.shape, "| type :", pluie.dtype, "| min :", pluie.min())

forme: (6, 7) | type : float64 | min : -999.0


In [21]:
# Cumul naïf par commune
print ("Cumul naïf (mm):", dict( zip ( communes, pluie.sum(axis=1))))

Cumul naïf (mm): {'Gonaïves': np.float64(293.0), 'Saint-Marc': np.float64(263.0), 'Cabaret': np.float64(335.0), 'Léogâne': np.float64(218.0), 'Petit-Goâve': np.float64(-696.0), 'Jacmel': np.float64(250.0)}


In [22]:
# Cumul naïf par commune. Enlever le Type. Utiliser ".tolist()"
print ("Cumul naïf (mm):", dict( zip ( communes, pluie.sum(axis=1).tolist())))

Cumul naïf (mm): {'Gonaïves': 293.0, 'Saint-Marc': 263.0, 'Cabaret': 335.0, 'Léogâne': 218.0, 'Petit-Goâve': -696.0, 'Jacmel': 250.0}


In [23]:
#Masque bolean : True  où la station a enregistre le code (-999)
sentinelle = pluie == -999
#Nombre de valeurs sentinelles, dans quelles communes
print("Nombre de valeurs sentinelles par commune :", int(sentinelle.sum()), "->", communes[sentinelle.any(axis=1)])

Nombre de valeurs sentinelles par commune : 1 -> ['Petit-Goâve']


In [25]:
#Remplacer ces valaeurs par NaN (valeur manquante) au lieu de les additioner  # NaN est par defaut a "float".
pluie[sentinelle] = np.nan

# Controle 1: la forme est inchangée, pluviometrie  non negative.
# option advance  : assert pluie.shape == (6,7) and  np.nanmin(pluie) >=0
assert pluie.shape == (6,7)
assert np.nanmin(pluie) >=0

In [26]:
print(pluie)
print(pluie.shape)

[[ 12.  18.  44.  96.  72.  31.  20.]
 [ 10.  14.  38.  88.  69.  27.  17.]
 [  8.  16.  51. 110.  83.  42.  25.]
 [  6.  11.  29.  75.  58.  24.  15.]
 [ 15.  21.  63. 124.  nan  48.  32.]
 [  9.  12.  35.  81.  64.  30.  19.]]
(6, 7)


In [27]:
# Nombre de jours avec une mesure valide, par commune. "~" est le NON logique pour les tableaux booléens, il inverse le masque, donc les cases valides  deviennes "True"
jours_valides = (~np.isnan(pluie)).max(axis=1)
#print("Nombre de jours avec une mesure valide, par commune :", jours_valides)

In [28]:
#Cumul observe en ignorant les  NaN
cumul_observe= np.nansum(pluie, axis=1)
print("Cumul observé (mm):", dict( zip ( communes, cumul_observe.tolist())))

Cumul observé (mm): {'Gonaïves': 293.0, 'Saint-Marc': 263.0, 'Cabaret': 335.0, 'Léogâne': 218.0, 'Petit-Goâve': 303.0, 'Jacmel': 250.0}


In [29]:
# Cumul sur 7 jours
cumul_observe_7j= cumul_observe/jours_valides * 7
print("Cumul observé sur 7 jours (mm):", dict( zip ( communes, cumul_observe_7j.tolist())))

Cumul observé sur 7 jours (mm): {'Gonaïves': 2051.0, 'Saint-Marc': 1841.0, 'Cabaret': 2345.0, 'Léogâne': 1526.0, 'Petit-Goâve': 2121.0, 'Jacmel': 1750.0}


In [30]:
# Pluie maximale d 'une journée par commune
pluie_max = np.nanmax(pluie, axis=1)
print("Pluie maximale d'une journée (mm):", dict( zip ( communes, pluie_max.tolist())))

Pluie maximale d'une journée (mm): {'Gonaïves': 96.0, 'Saint-Marc': 88.0, 'Cabaret': 110.0, 'Léogâne': 75.0, 'Petit-Goâve': 124.0, 'Jacmel': 81.0}


In [31]:
# Part de menages vivant en zone inondable (0,1)
exposition = pluie_df["part_menages_zone_inondable"].to_numpy()
print("Part de menages vivant en zone inondable :", dict( zip ( communes, exposition.tolist())))

Part de menages vivant en zone inondable : {'Gonaïves': 0.35, 'Saint-Marc': 0.22, 'Cabaret': 0.41, 'Léogâne': 0.3, 'Petit-Goâve': 0.38, 'Jacmel': 0.18}


In [32]:
# 1-  menages affectes  ; Indice d'exposition  ( pluie estime [cumul des 7 jours ] * part de menages exposes) , rapport  30 lignes (avec tableaux)
indice_exposition = (cumul_observe_7j * exposition)
print("Indice d'exposition :", dict( zip ( communes, indice_exposition.tolist())))

Indice d'exposition : {'Gonaïves': 717.8499999999999, 'Saint-Marc': 405.02, 'Cabaret': 961.4499999999999, 'Léogâne': 457.8, 'Petit-Goâve': 805.98, 'Jacmel': 315.0}


In [33]:
import numpy as np
import pandas as pd

# Lecture du fichier CSV
pluie_df = pd.read_csv("S05_pluie_7jours.csv")
# Liste des communes (liste Python, donc .index() fonctionne)
communes = pluie_df["commune"].tolist()
# Tableau NumPy 6 x 7 des pluies (j1 à j7)
pluie = pluie_df[["j1","j2","j3","j4","j5","j6","j7"]].to_numpy(dtype=float)
# Remplacer la valeur sentinelle -999 par NaN
pluie[pluie == -999] = np.nan

# Nombre de jours avec une mesure valide, par commune
jours_valides = (~np.isnan(pluie)).sum(axis=1)
# Cumul observé en ignorant les NaN
cumul_observe = np.nansum(pluie, axis=1)
# Cumul estimé sur 7 jours (jour manquant = moyenne des autres)
cumul_estim = cumul_observe / jours_valides * 7
# Pluie maximale d'une seule journée, par commune
pluie_max = np.nanmax(pluie, axis=1)
# Part des ménages en zone inondable (nom de colonne exact)
part_zone = pluie_df["part_menages_zone_inondable"].to_numpy()
# Indice d'exposition = pluie estimée (mm) x part des ménages exposés
indice = (cumul_estim * part_zone).round(1)

# Broadcasting : chaque ligne (6 x 7) divisée par son cumul en colonne (6 x 1)
part_pluie = pluie / cumul_observe.reshape(6, 1)
# Part du 4e jour (indice 3) dans le cumul, en %
part_jour4 = (part_pluie[:, 3] * 100).round(0)

# Contrôle 2 : recalcul à la main du cumul de Gonaïves
i = communes.index("Gonaïves")
a_la_main = sum(v for v in pluie[i] if not np.isnan(v))
assert a_la_main == cumul_observe[i], "Cumul de Gonaïves incorrect"

# Assembler la table finale
resultat = pd.DataFrame({
    "commune": communes,
    "jours_valides": jours_valides,
    "cumul_observe_mm": cumul_observe,
    "cumul_estime_7j_mm": cumul_estim.round(1),
    "max_journalier_mm": pluie_max,
    "indice_exposition": indice,
})
# Règle : vérification prioritaire si > 300 mm OU > 100 mm/jour
resultat["verification_prioritaire"] = (
    (resultat["cumul_estime_7j_mm"] > 300) | (resultat["max_journalier_mm"] > 100)
)
# Trier de l'indice le plus élevé au plus faible et renuméroter
resultat = resultat.sort_values("indice_exposition", ascending=False).reset_index(drop=True)
# Afficher la table finale
display(resultat)

# Tableau B : part du jour 4 (%)
tableau_b = pd.DataFrame({"commune": communes, "part_jour_4 (%)": part_jour4})
display(tableau_b)

,commune,jours_valides,cumul_observe_mm,cumul_estime_7j_mm,max_journalier_mm,indice_exposition,verification_prioritaire
0,Cabaret,7,335.0,335.0,110.0,137.4,True
1,Petit-Goâve,6,303.0,353.5,124.0,134.3,True
2,Gonaïves,7,293.0,293.0,96.0,102.6,False
3,Léogâne,7,218.0,218.0,75.0,65.4,False
4,Saint-Marc,7,263.0,263.0,88.0,57.9,False
5,Jacmel,7,250.0,250.0,81.0,45.0,False


,commune,part_jour_4 (%)
0,Gonaïves,33.0
1,Saint-Marc,33.0
2,Cabaret,33.0
3,Léogâne,34.0
4,Petit-Goâve,41.0
5,Jacmel,32.0
